# Cleanup: 03 — Incremental refresh artifacts

**Removes** resources introduced or used by `03_gdelt_incremental_refresh.ipynb`:
- Table `_sync_metadata` in `{BIGQUERY_DATASET}`
- Leftover `staging_{table}` tables in `{BIGQUERY_DATASET}` (US-CENTRAL1) and `{BIGQUERY_DATASET}_us` (US), for each table in `BIGQUERY_TABLES`
- Cloud Scheduler job **`gdelt-refresh-scheduler`** in **`{PROJECT_REGION}`** (see **Cloud Scheduler teardown** cell below; set `SKIP_CLOUD_SCHEDULER_DELETE=1` to skip)

**Does not** delete the Cloud Function **`gdelt-incremental-refresh`** (Gen2). Remove it in Cloud Console or with `gcloud functions delete` if you want that gone too.

**Does not** truncate merged rows in `gkg_partitioned`, `events_partitioned`, or `eventmentions_partitioned`. To wipe those and the rest of the warehouse, run `01_gdelt_data_prep_cleanup.ipynb`.

**IAM**: BigQuery permissions to delete tables in both datasets.

**Irreversible** for deleted tables (sync history and staging copies are removed).

**Config:** `{BIGQUERY_DATASET}` / `{BIGQUERY_DATASET}_us` match **`BIGQUERY_DATASET`** from **`config.py`** (loaded by the code cells below).

### Full reset order

1. **`02_gdelt_graph_create_cleanup.ipynb`** — Drop the BigQuery property graph while datasets still exist.
2. **`03_gdelt_incremental_refresh_cleanup.ipynb`** — Remove `_sync_metadata` and staging tables (optional if you immediately run step 4).
3. **`04_gdelt_data_profiling_cleanup.ipynb`** — Remove Dataplex scans and generated descriptions (dataset must still exist).
4. **`01_gdelt_data_prep_cleanup.ipynb`** — Delete `{BIGQUERY_DATASET}`, `{BIGQUERY_DATASET}_us`, and local exports (**always last**).

```mermaid
flowchart LR
  cleanup02[cleanup_02_drop_graph]
  cleanup03[cleanup_03_incremental_artifacts]
  cleanup04[cleanup_04_dataplex_and_descriptions]
  cleanup01[cleanup_01_datasets_and_local]
  cleanup02 --> cleanup01
  cleanup03 --> cleanup01
  cleanup04 --> cleanup01
```

In [ ]:
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
_cfg_dir = _cwd.parent if _cwd.name == "clean_up" else _cwd
if not (_cfg_dir / "config.py").is_file():
    _cfg_dir = _cwd
sys.path.insert(0, str(_cfg_dir))

from config import (
    GCP_PROJECT_ID,
    PROJECT_REGION,
    BIGQUERY_DATASET,
    BIGQUERY_TABLES,
    print_config,
    validate_config,
)

print_config()
if not validate_config():
    print("\n⚠️  Update config.py before proceeding.")
else:
    print("\n✅ Configuration loaded.")

In [ ]:
from google.cloud import bigquery

client = bigquery.Client(project=GCP_PROJECT_ID)
us_ds = f"{BIGQUERY_DATASET}_us"


def delete_table_safe(dataset_id, table_id, label=""):
    ref = bigquery.TableReference(
        bigquery.DatasetReference(GCP_PROJECT_ID, dataset_id), table_id
    )
    client.delete_table(ref, not_found_ok=True)
    suffix = f" {label}" if label else ""
    print(f"✅ Removed or absent:{suffix} {dataset_id}.{table_id}")


delete_table_safe(BIGQUERY_DATASET, "_sync_metadata", "[US-CENTRAL1]")

for table_name in BIGQUERY_TABLES:
    staging = f"staging_{table_name}"
    delete_table_safe(BIGQUERY_DATASET, staging, "[US-CENTRAL1]")
    delete_table_safe(us_ds, staging, "[US]")

print("\n✅ Incremental refresh cleanup finished.")

## Cloud Scheduler teardown

Notebook `03` / `cloud_function/deploy.sh` create **`gdelt-refresh-scheduler`** in **`PROJECT_REGION`**. The BigQuery cleanup cells above do **not** remove it automatically unless you run this section.

The next cell deletes that job via `gcloud`. Set **`SKIP_CLOUD_SCHEDULER_DELETE=1`** to skip (leave the job paused or enabled in the console).

In [ ]:
import os
import subprocess

SCHEDULER_JOB_NAME = "gdelt-refresh-scheduler"

if os.environ.get("SKIP_CLOUD_SCHEDULER_DELETE", "").strip().lower() in ("1", "true", "yes"):
    print("⏭️  SKIP_CLOUD_SCHEDULER_DELETE set — leaving Cloud Scheduler job in place.")
else:
    print(
        f"Deleting scheduler job {SCHEDULER_JOB_NAME} "
        f"(project={GCP_PROJECT_ID}, location={PROJECT_REGION})..."
    )
    proc = subprocess.run(
        [
            "gcloud",
            "scheduler",
            "jobs",
            "delete",
            SCHEDULER_JOB_NAME,
            f"--location={PROJECT_REGION}",
            f"--project={GCP_PROJECT_ID}",
            "--quiet",
        ],
        capture_output=True,
        text=True,
    )
    out = (proc.stderr or "") + (proc.stdout or "")
    if proc.returncode == 0:
        print("✅ Scheduler job deleted.")
    elif "not found" in out.lower() or "NOT_FOUND" in out:
        print("ℹ️  Scheduler job already absent.")
    else:
        print(out.strip() or f"gcloud exited {proc.returncode}")

## Optional sanity check

List tables remaining in the main dataset (expect core GDELT + graph tables if 01 was run).

In [ ]:
from google.cloud.exceptions import NotFound

try:
    tables = list(client.list_tables(BIGQUERY_DATASET))
    print(f"Tables in {BIGQUERY_DATASET}: {len(tables)}")
    for t in sorted(tables, key=lambda x: x.table_id):
        print(f"  - {t.table_id}")
except NotFound:
    print(f"Dataset {BIGQUERY_DATASET} not found (may already be removed).")